# 🐛 Pest Detection — YOLO26 Training
Same pipeline/style as the YOLOv8 / RT-DETR / YOLO11 notebook, now using **YOLO26** (Ultralytics' latest release — DFL-free head, MuSGD optimizer, STAL small-object assignment).

In [1]:
!pip install -U ultralytics -q

import ultralytics
ultralytics.checks()
print('\n✅ Setup complete!')

Ultralytics 8.4.114 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Setup complete ✅ (4 CPUs, 31.3 GB RAM, 7035.6/8062.4 GB disk)

✅ Setup complete!


In [2]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/gbsaiganesh
/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1
/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1/valid
/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1/test
/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1/train


In [3]:
!find /kaggle/input -maxdepth 3

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/gbsaiganesh
/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1


In [3]:
import glob

print(glob.glob("/kaggle/input/**/_annotations.coco.json", recursive=True))

['/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1/valid/_annotations.coco.json', '/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1/test/_annotations.coco.json', '/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1/train/_annotations.coco.json']


In [4]:
import json
import shutil
from pathlib import Path


DATASET_DIR = Path("/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1")

# Output folder
OUTPUT_DIR = Path("yolo_dataset")


def coco_to_yolo(split):
    img_dir = DATASET_DIR / split
    ann_file = img_dir / "_annotations.coco.json"

    if not ann_file.exists():
        print(f"❌ {ann_file} not found")
        return 0

    out_img = OUTPUT_DIR / split / "images"
    out_lbl = OUTPUT_DIR / split / "labels"

    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)

    with open(ann_file, "r") as f:
        coco = json.load(f)

    # image_id -> image information
    img_map = {img["id"]: img for img in coco["images"]}

    # image_id -> annotations
    ann_map = {}
    for ann in coco["annotations"]:
        ann_map.setdefault(ann["image_id"], []).append(ann)

    # Category mapping
    categories = sorted(
        [c for c in coco["categories"] if c["id"] != 0],
        key=lambda x: x["id"]
    )

    cat_map = {c["id"]: i for i, c in enumerate(categories)}

    print(f"\n{split.upper()} CLASSES:")
    print([c["name"] for c in categories])

    converted = 0
    skipped = 0

    for img_id, img in img_map.items():

        image_name = img["file_name"]
        W = img["width"]
        H = img["height"]

        src = img_dir / image_name
        dst = out_img / image_name

        if not src.exists():
            skipped += 1
            continue

        shutil.copy2(src, dst)

        label_file = out_lbl / (Path(image_name).stem + ".txt")

        with open(label_file, "w") as lf:

            for ann in ann_map.get(img_id, []):

                if ann["category_id"] not in cat_map:
                    continue

                cls = cat_map[ann["category_id"]]

                x, y, w, h = ann["bbox"]

                xc = (x + w / 2) / W
                yc = (y + h / 2) / H
                wn = w / W
                hn = h / H

                lf.write(f"{cls} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}\n")

        converted += 1

    print(f"Converted: {converted} | Skipped: {skipped}")
    return converted


for split in ["train", "valid", "test"]:
    coco_to_yolo(split)


TRAIN CLASSES:
['Aphids', 'Bacterial Rot', 'Bacterial Spot', 'Blossom end rot', 'Early Blight', 'Fruit Crack', 'Fruit Damage', 'Leafhoppers and Jassids', 'Pest Damage', 'Potassium Deficiency', 'Viral', 'bactarial canker', 'bug', 'caterpilla', 'late blight', 'leafminer', 'powedry_mildew', 'spodoptera frugiperda', 'tuta', 'whiteflies']
Converted: 4353 | Skipped: 0

VALID CLASSES:
['Aphids', 'Bacterial Rot', 'Bacterial Spot', 'Blossom end rot', 'Early Blight', 'Fruit Crack', 'Fruit Damage', 'Leafhoppers and Jassids', 'Pest Damage', 'Potassium Deficiency', 'Viral', 'bactarial canker', 'bug', 'caterpilla', 'late blight', 'leafminer', 'powedry_mildew', 'spodoptera frugiperda', 'tuta', 'whiteflies']
Converted: 636 | Skipped: 0

TEST CLASSES:
['Aphids', 'Bacterial Rot', 'Bacterial Spot', 'Blossom end rot', 'Early Blight', 'Fruit Crack', 'Fruit Damage', 'Leafhoppers and Jassids', 'Pest Damage', 'Potassium Deficiency', 'Viral', 'bactarial canker', 'bug', 'caterpilla', 'late blight', 'leafminer'

In [5]:
import os

for root, dirs, files in os.walk("yolo_dataset"):
    print(root)
    for f in files[:5]:
        print("   ", f)

yolo_dataset
yolo_dataset/valid
yolo_dataset/valid/images
    IMG_4459_jpg.rf.9804774ac9e35f544740e95d47e6d369.jpg
    IMG_6432_jpg.rf.7529f884043727da063381820c61122c.jpg
    IMG_4963_jpg.rf.2be36dd5d9755615161df4773891a059.jpg
    IMG_6506_jpg.rf.a718bd9b47098589712c09ae44141e05.jpg
    r-549-_jpg.rf.9fdda992c9a9bea8f3df9409ce3f341a.jpg
yolo_dataset/valid/labels
    IMG20240305122257_01_jpg.rf.1d9838bb7a1c03b486db8b3bc1b38cd4.txt
    HA-101-_jpg.rf.5b23a340b541f807da173d9b2dd47cb2.txt
    29Watermelon_Mosaic_jpg.rf.335e60cd07592c26d34147fb07738b79.txt
    201305060647260672_jpg.rf.ad7d2627a6c9444f70c30ab55c8b41bb.txt
    TU-57-_jpg.rf.8c52a896fef13e62d886a077aa290a36.txt
yolo_dataset/test
yolo_dataset/test/images
    37Pepper_BacterialSpot_jpg.rf.262ef802d1246d87e223f4f5e084497c.jpg
    cotton-leafhoppers-and-jassids-2-farmonaut_jpg.rf.78b003c929ed61e802b312505a4d7e5a.jpg
    Bacterial-Canker_Tomato_scanned-slide-1-_jpg.rf.5e18a5111613331bbf80b722ffbda277.jpg
    d-94-_png.rf.fe866ae

In [6]:
from pathlib import Path

for split in ["train", "valid", "test"]:
    img_count = len(list(Path(f"yolo_dataset/{split}/images").glob("*")))
    lbl_count = len(list(Path(f"yolo_dataset/{split}/labels").glob("*.txt")))

    print(f"{split}")
    print(f"Images : {img_count}")
    print(f"Labels : {lbl_count}\n")

train
Images : 4353
Labels : 4353

valid
Images : 636
Labels : 636

test
Images : 181
Labels : 181



In [7]:
import json

with open("/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1/train/_annotations.coco.json") as f:
    coco = json.load(f)

print("Number of classes:", len(coco["categories"]))

for c in coco["categories"]:
    print(c["id"], c["name"])

Number of classes: 21
0 Phase2-Phase3-tomato-Spodop-iEGw
1 Aphids
2 Bacterial Rot
3 Bacterial Spot
4 Blossom end rot
5 Early Blight
6 Fruit Crack
7 Fruit Damage
8 Leafhoppers and Jassids
9 Pest Damage
10 Potassium Deficiency
11 Viral
12 bactarial canker
13 bug
14 caterpilla
15 late blight
16 leafminer
17 powedry_mildew
18 spodoptera frugiperda
19 tuta
20 whiteflies


In [8]:
import json
from pathlib import Path

# ---- YOUR DATASET PATH ----
dataset_path = "/kaggle/input/datasets/gbsaiganesh/leaf-pests-dataset1"

# Read COCO annotation
ann_file = f"{dataset_path}/train/_annotations.coco.json"

with open(ann_file, "r") as f:
    coco = json.load(f)

# Extract categories
categories = sorted(coco["categories"], key=lambda x: x["id"])

# Build YAML text
yaml_text = """path: /kaggle/working/yolo_dataset
train: train/images
val: valid/images
test: test/images

names:
"""

for i, cat in enumerate(categories):
    yaml_text += f"  {i}: {cat['name']}\n"

# SAVE THE FILE
yaml_path = "/kaggle/working/data.yaml"
Path(yaml_path).write_text(yaml_text)

print("✅ data.yaml saved at:", yaml_path)
print("\n----- YAML CONTENT -----\n")
print(yaml_text)

✅ data.yaml saved at: /kaggle/working/data.yaml

----- YAML CONTENT -----

path: /kaggle/working/yolo_dataset
train: train/images
val: valid/images
test: test/images

names:
  0: Phase2-Phase3-tomato-Spodop-iEGw
  1: Aphids
  2: Bacterial Rot
  3: Bacterial Spot
  4: Blossom end rot
  5: Early Blight
  6: Fruit Crack
  7: Fruit Damage
  8: Leafhoppers and Jassids
  9: Pest Damage
  10: Potassium Deficiency
  11: Viral
  12: bactarial canker
  13: bug
  14: caterpilla
  15: late blight
  16: leafminer
  17: powedry_mildew
  18: spodoptera frugiperda
  19: tuta
  20: whiteflies



In [9]:
import torch
from ultralytics import YOLO


print("CUDA available:", torch.cuda.is_available())
print("Number of GPUs:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

CUDA available: True
Number of GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [10]:
from ultralytics import YOLO

model = YOLO("yolo26s.pt")

results = model.train(
    data="/kaggle/working/data.yaml",

    epochs=50,
    imgsz=800,
    batch=32,

    device=[0, 1],
    workers=8,

    optimizer="auto",   # YOLO26 uses MuSGD by default via optimizer='auto'
    lr0=0.001,
    lrf=0.01,

    weight_decay=0.0005,

    cos_lr=True,

    warmup_epochs=5,

    patience=30,

    augment=True,

    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.2,

    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    project="/kaggle/working/runs/detect",
    name="YOLO26s_improved",
    exist_ok=True
)

Ultralytics 8.4.114 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.2, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo26s.pt, m

In [11]:
metrics = model.val()

print("========== FINAL RESULTS ==========")
print("mAP@50      :", round(metrics.box.map50, 4))
print("mAP@50-95   :", round(metrics.box.map, 4))
print("Precision   :", round(metrics.box.mp, 4))
print("Recall      :", round(metrics.box.mr, 4))

Ultralytics 8.4.114 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26s summary (fused): 122 layers, 9,473,307 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2961.4±1613.9 MB/s, size: 799.7 KB)
val: Scanning /kaggle/working/yolo_dataset/valid/labels.cache... 636 images, 169 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 636/636 74.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 40/40 2.1it/s 19.0s0.4s
                   all        636       1195      0.491       0.52      0.497      0.248
Phase2-Phase3-tomato-Spodop-iEGw          4         20      0.544       0.25      0.254      0.139
        Bacterial Spot          2          2          1      0.856      0.995      0.316
          Fruit Damage          4          4          0          0     0.0472     0.0175
  Potassium Deficiency          1          1          0          0          0          0
             

In [12]:
import os

for split in ["train", "valid", "test"]:

    img_path = f"/kaggle/working/yolo_dataset/{split}/images"
    lbl_path = f"/kaggle/working/yolo_dataset/{split}/labels"

    print("\n", split.upper())
    print("Images :", len(os.listdir(img_path)) if os.path.exists(img_path) else 0)
    print("Labels :", len(os.listdir(lbl_path)) if os.path.exists(lbl_path) else 0)


 TRAIN
Images : 4353
Labels : 4353

 VALID
Images : 636
Labels : 636

 TEST
Images : 181
Labels : 181
